# 02: Volt-Watt and Volt-VAr conformance and curtailment

This notebook keeps three questions separate: **maximum-output or required-Q conformance**, **whether the observation was informative about an active response**, and **counterfactual-supported curtailed energy**. 

The 10% site threshold is a project reporting convention, not a tolerance granted by AS/NZS 4777.2.

## Interpretation contract

- Volt-Watt conformance asks whether measured P exceeded the voltage-dependent maximum. It does not require a counterfactual.
- Volt-Watt response support asks whether measured P already proved a violation or the counterfactual showed enough solar resource to test the response.
- Curtailment energy is calculated only on intervals with a counterfactual. Missing counterfactuals are reported, never replaced with zero.
- Volt-VAr conformance uses the Stage 2 capability-assessable denominator. Low-power intervals excluded by Section 2.6 (Figure 2.1) of AS/NZ 4777.2.2020 implementation are reported separately.

In [1]:
%matplotlib inline

import importlib
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Locate the repository.
HERE = Path.cwd().resolve()

REPO_ROOT = next(
    (
        path
        for path in (HERE, *HERE.parents)
        if (path / "bms_sa_review").is_dir()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Cannot find the repository root containing bms_sa_review. "
        f"Current directory: {HERE}"
    )


# Expose all import styles currently used in the repository.
PACKAGE_ROOT = REPO_ROOT / "bms_sa_review"
SHARED = PACKAGE_ROOT / "shared"
LIB = PACKAGE_ROOT / "data_query" / "lib"

for path in (REPO_ROOT, PACKAGE_ROOT, SHARED, LIB):
    path_string = str(path)

    if path_string not in sys.path:
        sys.path.insert(0, path_string)


# Project imports.
from bms_sa_review.shared.aws_config import aq

import analysis_contract as contract
import conformance_queries as cq
import conformance_metrics as cm
import conformance_plots as cp


# Reload local analytical modules during notebook development.
for module in (contract, cq, cm, cp):
    importlib.reload(module)


pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

print("Repository root:", REPO_ROOT)
print("Package root:", PACKAGE_ROOT)
print("Shared modules:", SHARED)
print("Analysis library:", LIB)

Repository root: C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA
Package root: C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA\bms_sa_review
Shared modules: C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA\bms_sa_review\shared
Analysis library: C:\Users\z3553082\OneDrive - UNSW\Documents\GitHub\CICCADA\bms_sa_review\data_query\lib


In [2]:
# Running for FLEX included tables:

FLEX_INCLUDED_TABLES = dict(contract.TABLES)

FLEX_INCLUDED_TABLES.update({
    "structured_data": "structured_data_v2_flex_included",
    "all_uncurtailedpv": "all_uncurtailedpv_v2_flex_included",
    "conformance_voltwatt": "conformance_voltwatt_v2_flex_included",
    "conformance_voltwattghi": "conformance_voltwattghi_v2_flex_included",
    "conformance_voltvar": "conformance_voltvar_v2_flex_included",
})

CONFIG = contract.AnalysisConfig(
    years=(2024, 2025),
    flex_selection="include",
    rating_basis="ac_capacity_kw",
    empirical_limit_basis="s_99",
    voltage_aggregation="avg",
    capability_profile="review_corrected",
    day_night="all",
    tables=FLEX_INCLUDED_TABLES,
).validate()

display(contract.manifest(CONFIG))

,setting,value
0,database,solar_analytics_iceberg
1,years,"2024, 2025"
2,interval_minutes,5.0
3,site_nonconf_threshold,0.1
4,minimum_site_intervals,1
5,flex_selection,include
6,rating_basis,ac_capacity_kw
7,empirical_limit_basis,s_99
8,voltage_aggregation,avg
9,capability_profile,review_corrected


## 1. Inputs, stored provenance, and metadata integrity

In [3]:
inventory = cq.fetch_inventory(aq, CONFIG)
stored_provenance = cq.fetch_stored_provenance(aq, CONFIG)
display(inventory)
display(stored_provenance)

meta = cq.fetch_metadata(aq, CONFIG)
meta_conflicts = cm.validate_metadata(meta)
display(meta_conflicts)
assert not meta['site_id'].duplicated().any()

,logical_name,table_name,n_rows,n_sites,first_year,last_year
0,structured_data,structured_data_v2_flex_included,871655350,15993,2024,2025
1,all_uncurtailedpv,all_uncurtailedpv_v2_flex_included,504416916,15888,2024,2025
2,conformance_voltvar,conformance_voltvar_v2_flex_included,15653733,16148,2024,2025
3,conformance_voltwatt,conformance_voltwatt_v2_flex_included,15653733,16148,2024,2025
4,conformance_voltwattghi,conformance_voltwattghi_v2_flex_included,15653733,16148,2024,2025


,table_name,normalization_basis,voltage_aggregation,flex_selection,n_rows,counterfactual_cap_basis,rating_basis,empirical_limit_basis,capability_profile
0,structured_data_v2_flex_included,s_99,avg,include,871655350,<NA>,<NA>,<NA>,<NA>
1,all_uncurtailedpv_v2_flex_included,s_99,<NA>,<NA>,504416916,none,<NA>,<NA>,<NA>
2,conformance_voltwatt_v2_flex_included,<NA>,avg,include,15653733,<NA>,ac_capacity_kw,<NA>,<NA>
3,conformance_voltwattghi_v2_flex_included,<NA>,avg,include,15653733,<NA>,ac_capacity_kw,<NA>,<NA>
4,conformance_voltvar_v2_flex_included,<NA>,avg,include,15653733,<NA>,ac_capacity_kw,s_99,review_corrected


,field,sites_with_multiple_values
0,n_state_values,0
1,n_dnsp_values,0
2,n_oem_values,0
3,n_ac_capacity_values,0
4,n_s99_values,0


In [ ]:
population_funnel = cq.fetch_population_funnel(aq, CONFIG)
display(population_funnel.T.rename(columns={0: 'n_sites'}))

## 2. Volt-Watt maximum-output conformance

**Question:** when voltage selected a reduced maximum active-power level, did measured P exceed that permitted maximum (including the stored ±4% band)? A below-ceiling observation satisfies this maximum-output test but does not, by itself, demonstrate active curtailment.

In [ ]:
#### Parameters presented below ####
# n_sites = The number of distinct sites with at least one evaluated Volt-Watt interval.
# n_intervals = The total number of five-minute site intervals where *average* site voltage was greater than 253 V.
# n_nonconforming_intervals = The number of voltage-exposed intervals where measured active power exceeded the permitted Volt-Watt maximum: P_measured > P_VW_curve(V) + 0.04 × ac_capacity_kw
# interval_nonconf_pct = 100 × n_nonconforming_intervals / n_intervals

#### Agreggations ####
# fleet_conformant_pct:
## First, a nonconformance fraction is calculated separately for each site: site nonconformance fraction = site nonconforming intervals / site exposed intervals
## The project rule then classifies a site as conformant when: site nonconformance fraction <= 10%
## fleet_conformant_pct is the percentage of evaluated sites satisfying that rule: fleet_conformant_pct = 100 × conformant sites / evaluated sites

# fleet_any_nonconf_pct
## The percentage of evaluated sites with at least one nonconforming interval:
## fleet_any_nonconf_pct = 100 × sites with at least one nonconforming interval / evaluated sites

In [ ]:
vw_site_year = cq.fetch_vw_site_year(aq, CONFIG)
vw_sites = cm.prepare_vw_conformance(vw_site_year, CONFIG)
vw_table_sites = vw_site_year["site_id"].nunique()
vw_result = cm.fleet_result(
    vw_sites,
    "Volt-Watt maximum-output conformance",
    "Did measured P exceed the voltage-dependent maximum?",
    "voltage-exposed intervals (V > 253 V)",
    table_site_count=vw_table_sites,
)
display(vw_result)
display(vw_sites.sort_values('nonconf_frac', ascending=False).head(20))

### Normalised magnitude of Volt-Watt nonconformance

The site-conformance result above measures the frequency of nonconformance. The following metric additionally measures its magnitude.

For each site, nonconforming kW samples are converted to Wh and normalised by AC nameplate capacity and the number of voltage-exposed intervals:

`normalised NC = nonconformance Wh / (ac_capacity_kw × exposed intervals)`

The resulting unit is Wh per kW-nameplate per exposed interval. This is a severity metric and does not replace the 10% frequency-based site classification.

In [ ]:
vw_sites_normalized = cm.add_vw_normalized_nonconformance(
    vw_sites,
    meta,
    CONFIG.interval_h,
    capacity_col="ac_capacity_kw",
)

display(
    vw_sites_normalized[
        [
            "site_id",
            "nonconf_count",
            "denominator_count",
            "nonconformance_wh",
            "normalized_nonconformance_wh_per_kw_interval",
        ]
    ]
    .sort_values(
        "normalized_nonconformance_wh_per_kw_interval",
        ascending=False,
    )
    .head(20)
)

In [ ]:
vw_sites_normalized[vw_sites_normalized['nonconformant'] == True]

## 3. Volt-Watt response support and counterfactual coverage

This is a narrower evidentiary population, not a replacement compliance population. An interval is response-supported when actual P proves a violation or the counterfactual exceeds the permitted maximum.

In [ ]:
#### Parameters presented below ####

# exposed_intervals = All five-minute intervals where average site voltage was greater than 253 V

# response_supported_intervals:
## An exposed interval is response-supported when either:
## measured P exceeded the permitted Volt-Watt maximum, establishing a definite violation; or
## counterfactual P exceeded the permitted maximum, showing that sufficient solar power was available to test the response.

# missing_counterfactual_intervals = The number of voltage-exposed intervals for which all_uncurtailedpv_v2 supplied no matching counterfactual estimate

# response_supported_pct_of_exposed = 100 × response-supported intervals / exposed intervals
# missing_counterfactual_pct_of_exposed = 100 × missing-counterfactual intervals / exposed intervals 

#### Aggregations ####
# interval_nonconf_pct = 100 × nonconforming intervals / response-supported intervals
### This is not be presented as the fleet-wide Volt-Watt nonconformance rate.
### It is high because:
### - every measured violation enters the response-supported population;
### - a below-limit observation enters only when the counterfactual proves that enough solar power was available to test the response. 

# fleet_conformant_pct:
## For each response-supported site:
## site response-supported nonconformance fraction = site nonconforming intervals / site response-supported intervals
## The same project threshold is then applied:
## conformant if site fraction <= 10% 
## (This is not the primary fleet conformance result. It is a narrower result among sites where at least one informative response verdict was available.)

# fleet_any_nonconf_pct
## The percentage of response-supported sites with at least one directly observed violation:
## fleet_any_nonconf_pct = 100 × response-supported sites with any violation / all response-supported sites

In [ ]:
vwg_site_year = cq.fetch_vw_response_site_year(aq, CONFIG)
vwg_sites = cm.prepare_vw_response(vwg_site_year, CONFIG)
vwg_table_sites = vwg_site_year["site_id"].nunique()
vwg_result = cm.fleet_result(
    vwg_sites,
    "Volt-Watt response-supported result",
    "Was enough solar power available to test the response, or was a violation directly observed?",
    "response-supported intervals",
    table_site_count=vwg_table_sites,
)
display(cm.coverage_summary(vwg_site_year))
display(vwg_result)
membership = vw_sites[['site_id']].merge(vwg_sites[['site_id']], on='site_id', how='outer', indicator=True)
display(membership['_merge'].value_counts().rename_axis('membership').to_frame('n_sites'))

In [ ]:
vw_magnitude_sites = cm.vw_nonconformance_magnitude(
    vw_sites,
    meta,
    CONFIG.interval_h,
    capacity_col="ac_capacity_kw",
)

vw_magnitude_fleet = pd.DataFrame([{
    "n_sites": vw_magnitude_sites["site_id"].nunique(),

    "assessed_intervals":
        int(vw_magnitude_sites["denominator_count"].sum()),

    "nonconforming_intervals":
        int(vw_magnitude_sites["nonconf_count"].sum()),

    "nonconforming_kwh":
        vw_magnitude_sites["nonconforming_kwh"].sum(),

    "fleet_weighted_nonconforming_kwh_per_kw_interval": (
        vw_magnitude_sites["nonconforming_kwh"].sum()
        / vw_magnitude_sites[
            "capacity_interval_exposure_kw"
        ].sum()
    ),
}])

display(vw_magnitude_fleet)

display(
    vw_magnitude_sites[
        [
            "site_id",
            "nonconf_count",
            "denominator_count",
            "nonconforming_kwh",
            "ac_capacity_kw",
            "normalized_nonconformance_kwh_per_kw_interval",
        ]
    ]
    .sort_values("nonconforming_kwh", ascending=False)
    .head(20)
)

In [ ]:
vw_magnitude_breakdowns = {}

for group in ("state", "dnsp", "oem", "install_year"):
    vw_magnitude_breakdowns[group] = (
        cm.vw_nonconformance_magnitude_breakdown(
            vw_sites,
            meta,
            group,
            CONFIG.interval_h,
            capacity_col="ac_capacity_kw",
            min_sites=20,
        )
    )

    print(f"\nVolt-Watt nonconformance magnitude by {group}")
    display(vw_magnitude_breakdowns[group])

### Minimum intervals sensitivity

minimum_intervals is a site-inclusion threshold. It does not replace or change the 10% rule. For example:
- minimum 1: include any site with at least one evaluated interval;
- minimum 10: include only sites with at least 10 evaluated intervals;
- minimum 100: include only sites with at least 100 evaluated intervals.

After applying that inclusion criterion, the same site rule remains: {conformant if >= 10%} 


This matters because classifications based on very few observations are unstable:
- 1 violation out of 1 interval = 100% and nonconformant;
- 1 violation out of 10 intervals = 10% and conformant;
- 2 violations out of 10 intervals = 20% and nonconformant;
- 10 violations out of 100 intervals = 10% and conformant.

In [ ]:
vw_sensitivity = cm.minimum_interval_sensitivity(
    vw_site_year, 'nonconf_count', 'exposed_count', CONFIG)
vwg_sensitivity = cm.minimum_interval_sensitivity(
    vwg_site_year, 'nonconf_count', 'response_supported_count', CONFIG)
display(vw_sensitivity)
display(vwg_sensitivity)
cp.plot_sensitivity(vw_sensitivity, 'Volt-Watt maximum-output result')
cp.plot_sensitivity(vwg_sensitivity, 'Volt-Watt response-supported result')

## 4. Volt-Watt curtailed energy

Energy is reported only for counterfactual-covered intervals where potential P exceeded the permitted maximum and measured P did not. Five-minute kW samples are multiplied by 1/12 to obtain kWh.

The Stage 1 counterfactual table is a quality-gated modelling product rather than a complete copy of the telemetry time series. A counterfactual is written only when the site passes the MAPE gate, a structured-data and clear-sky reference is available, the interval passes the irradiance and generation filters, and a sufficiently trained time-of-day model is available. Consequently, an exposed interval may retain valid voltage and measured-power telemetry but have no counterfactual estimate. Missing counterfactuals reduce the coverage of response-opportunity and curtailed-energy analyses; they do not remove the interval from the basic maximum-output conformance analysis.

In [ ]:
100 * ((4646.429809+3537.847467) / (1.149299e+08+1.035444e+08))

In [ ]:
vw_energy_site_year = cq.fetch_vw_energy_site_year(aq, CONFIG)
vw_energy_sites, vw_energy_summary = cm.energy_summary(
    vw_energy_site_year, CONFIG.interval_h, 'Volt-Watt counterfactual curtailment')
display(vw_energy_summary)
display(vw_energy_sites.sort_values('curtailed_kwh', ascending=False).head(20))

In [ ]:
vw_legacy_energy_site_year = cq.fetch_vw_legacy_energy_site_year(
    aq,
    CONFIG,
)

vw_legacy_energy_sites, vw_legacy_energy_summary = (
    cm.legacy_exposed_energy_summary(
        vw_legacy_energy_site_year,
        CONFIG.interval_h,
    )
)

display(vw_legacy_energy_summary)

display(
    vw_legacy_energy_sites
    .sort_values("curtailed_kwh", ascending=False)
    .head(20)
)

## 5. Volt-VAr conformance

**Question:** among intervals where the selected Figure 2.1 capability implementation provides a quantified minimum, was measured Q within the required/capability-clamped band? The project failure definition is adverse + inactive + significant shortfall. Near-conformant and surplus are shown separately.

In [ ]:
vvar_site_year = cq.fetch_vvar_site_year(aq, CONFIG)
vvar_sites = cm.prepare_vvar_conformance(vvar_site_year, CONFIG)
vvar_result = cm.fleet_result(
    vvar_sites, 'Volt-VAr capability-conditioned conformance',
    'Was measured Q within the required and capability-clamped band?',
    'capability-assessable intervals',
)
vvar_bands = cm.vvar_band_summary(vvar_site_year)
vvar_energy_site_year, vvar_energy_summary = cm.vvar_curtailment_summary(
    vvar_site_year, CONFIG.interval_h)
display(vvar_result)
display(vvar_bands)
display(vvar_energy_summary)
cp.plot_vvar_bands(vvar_bands);

## 6. State, DNSP, OEM, installation-year and time breakdowns

In [ ]:
breakdowns = {}
for mechanism, frame in [('Volt-Watt', vw_sites), ('Volt-VAr', vvar_sites)]:
    for group in ('state', 'dnsp', 'oem', 'install_year'):
        key = (mechanism, group)
        breakdowns[key] = cm.group_breakdown(frame, meta, group, min_sites=20)
        print(f'\n{mechanism} by {group}')
        display(breakdowns[key])

cp.plot_group_breakdown(breakdowns[('Volt-Watt','state')], 'state', 'Volt-Watt by state')
cp.plot_group_breakdown(breakdowns[('Volt-VAr','state')], 'state', 'Volt-VAr by state')

vw_energy_breakdowns = {g: cm.energy_group_breakdown(vw_energy_sites, meta, g, min_sites=20)
                        for g in ('state','dnsp','oem','install_year')}
vvar_energy_breakdowns = {g: cm.energy_group_breakdown(vvar_energy_site_year, meta, g, min_sites=20)
                          for g in ('state','dnsp','oem','install_year')}
print('\nVolt-Watt curtailed energy by DNSP')
display(vw_energy_breakdowns['dnsp'])
print('\nVolt-VAr curtailed energy by DNSP')
display(vvar_energy_breakdowns['dnsp'])

### Volt-Watt nonconformance magnitude by fleet group

The following breakdowns supplement the frequency-based site classifications with the normalised magnitude of excess active power.

The mean and median site metrics give each site equal weight. The fleet-weighted metric divides total nonconformance Wh by total capacity-interval exposure, so systems with more exposed observations and greater nameplate capacity contribute proportionally more weight.

In [ ]:
vw_normalized_breakdowns = {}

for group in ("state", "dnsp", "oem", "install_year"):
    vw_normalized_breakdowns[group] = (
        cm.vw_normalized_group_breakdown(
            vw_sites,
            meta,
            group,
            CONFIG.interval_h,
            capacity_col="ac_capacity_kw",
            min_sites=20,
        )
    )

    print(f"\nVolt-Watt normalised nonconformance by {group}")
    display(vw_normalized_breakdowns[group])

### Monthly thread

In [ ]:
monthly_vw = cm.monthly_rates(cq.fetch_monthly_fleet(aq, CONFIG, 'vw'))
monthly_vwg = cm.monthly_rates(cq.fetch_monthly_fleet(aq, CONFIG, 'vw_response'))
monthly_vvar = cm.monthly_rates(cq.fetch_monthly_fleet(aq, CONFIG, 'vvar'))
display(monthly_vw)
cp.plot_monthly_rates(('Volt-Watt maximum-output', monthly_vw),
                      ('Volt-Watt response-supported', monthly_vwg),
                      ('Volt-VAr', monthly_vvar));

## 7. Legacy reconciliation and reporting-ready tally

In [ ]:
legacy_by_year = cq.fetch_legacy_population_by_year(aq, CONFIG)
legacy_membership = cq.fetch_original_v2_membership(aq, CONFIG)
display(legacy_by_year)
display(legacy_membership)
print('Do not add yearly site counts: sites can appear in both years.')

In [ ]:
headline_results = cm.reporting_tally(vw_result, vwg_result, vvar_result)
display(headline_results)
cp.plot_fleet_summary(headline_results);

reporting_notes = pd.DataFrame([
    ['Volt-Watt maximum-output', 'Primary standards-oriented maximum-output test', 'Counterfactual not required; below ceiling does not prove active response'],
    ['Volt-Watt response-supported', 'Evidence that the response had an opportunity to operate', 'Narrower GHI-supported population; not the primary compliance denominator'],
    ['Volt-Watt curtailed energy', 'Estimated generation displaced', 'Only counterfactual-covered intervals'],
    ['Volt-VAr conformance', 'Required Q conditional on implemented capability floor', 'S_rated is proxied by the recorded rating basis'],
    ['Volt-VAr curtailment', 'Headline only here; causal evidence is in notebook 03', 'S_99 is empirical, not verified manufacturer S_rated'],
], columns=['result', 'question_answered', 'principal_limitation'])
display(reporting_notes)

# Scratch

In [ ]:
SAI = "solar_analytics_iceberg"

In [ ]:
population_reconciliation = aq("""
WITH stage2_population AS (
    SELECT
        site_id,
        year
    FROM conformance_voltwattghi_v2
    WHERE year = 2024
    GROUP BY site_id, year
    HAVING SUM(
        COALESCE(total_count, 0)
        - COALESCE(null_uncurtailed_P_count, 0)
    ) > 0
),

direct_energy_population AS (
    SELECT DISTINCT
        u.site_id,
        YEAR(u.t_stamp + INTERVAL '10' HOUR) AS year
    FROM all_uncurtailedpv_v2 u
    INNER JOIN structured_data_v2 sd
        ON sd.site_id = u.site_id
       AND sd.t_stamp = u.t_stamp
    WHERE YEAR(u.t_stamp + INTERVAL '10' HOUR) = 2024
      AND sd.V > 253
),

membership AS (
    SELECT
        COALESCE(s.site_id, d.site_id) AS site_id,
        CASE
            WHEN s.site_id IS NOT NULL AND d.site_id IS NOT NULL
                THEN 'both'
            WHEN s.site_id IS NOT NULL
                THEN 'stage2_only'
            ELSE 'direct_energy_only'
        END AS membership
    FROM stage2_population s
    FULL OUTER JOIN direct_energy_population d
        ON s.site_id = d.site_id
       AND s.year = d.year
)

SELECT
    membership,
    COUNT(*) AS n_sites
FROM membership
GROUP BY membership
ORDER BY membership
""", database=SAI)

display(population_reconciliation)